In [1]:
!pip install peft -q

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

import re, random, glob, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import Wav2Vec2Model, Wav2Vec2Processor, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, roc_auc_score
from scipy.signal import fftconvolve
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

/data/liharrison/miniconda3/envs/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


In [3]:
# Build dataframe from concatenated audio streams (skip mild)
records = []

for f in sorted(glob.glob(os.path.join(BASE_DIR, "data", "concat_control", "*.wav"))):
    m = re.match(r"ctrl_gt(\d+)_spk(\d+)\.wav", os.path.basename(f))
    records.append({
        "abs_path": f, "label": "control", "label_id": 0,
        "severity": "none", "gt_idx": int(m.group(1)), "speaker_id": int(m.group(2)),
    })

for sev in ["moderate", "severe"]:
    for f in sorted(glob.glob(os.path.join(BASE_DIR, "data", "concat_dysfluent", sev, "*.wav"))):
        m = re.match(r"dys_\w+_gt(\d+)_spk(\d+)\.wav", os.path.basename(f))
        records.append({
            "abs_path": f, "label": "dysfluent", "label_id": 1,
            "severity": sev, "gt_idx": int(m.group(1)), "speaker_id": int(m.group(2)),
        })

df = pd.DataFrame(records)
print(f"Total streams (no mild): {len(df)}")
print(df["severity"].value_counts().to_string())

# 80/20 test split, then 85/15 train/val from the train portion
train_full_df, test_df = train_test_split(
    df, test_size=0.2, random_state=SEED, stratify=df["label_id"]
)
train_df, val_df = train_test_split(
    train_full_df, test_size=0.15, random_state=SEED, stratify=train_full_df["label_id"]
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"\nTrain: {len(train_df)} streams ({train_df['label_id'].mean():.1%} dysfluent)")
print(f"Val:   {len(val_df)} streams ({val_df['label_id'].mean():.1%} dysfluent)")
print(f"Test:  {len(test_df)} streams ({test_df['label_id'].mean():.1%} dysfluent)")

Total streams (no mild): 440
severity
none        220
moderate    110
severe      110

Train: 299 streams (50.2% dysfluent)
Val:   53 streams (49.1% dysfluent)
Test:  88 streams (50.0% dysfluent)


In [4]:
TARGET_SR = 16000
WINDOW_SEC = 15
WINDOW_SAMPLES = WINDOW_SEC * TARGET_SR  # 240,000
SAMPLES_PER_STREAM = 3

def augment_waveform(wav):
    """Apply random waveform augmentations. Input/output: 1-D tensor."""
    if random.random() < 0.5:
        factor = random.uniform(0.9, 1.1)
        new_len = int(len(wav) / factor)
        wav = F.interpolate(wav.view(1, 1, -1), size=new_len, mode="linear", align_corners=False).squeeze()
    if random.random() < 0.5:
        snr_db = random.uniform(10, 20)
        sig_pow = wav.pow(2).mean()
        noise = torch.randn_like(wav) * (sig_pow / (10 ** (snr_db / 10))).sqrt()
        wav = wav + noise
    if random.random() < 0.5:
        wav = wav * (10 ** (random.uniform(-6, 6) / 20))
    if random.random() < 0.3:
        rt60 = random.uniform(0.2, 0.8)
        n = int(TARGET_SR * rt60)
        t = torch.arange(n, dtype=torch.float32) / TARGET_SR
        rir = torch.randn(n) * torch.exp(-3.0 * t / rt60)
        rir = rir / rir.abs().max()
        out = fftconvolve(wav.numpy(), rir.numpy(), mode="full")[:len(wav)]
        wav = torch.from_numpy(out.astype(np.float32))
    return wav

class ConcatSpeechDataset(Dataset):
    """Dataset that preloads concatenated streams and samples fixed-length windows."""
    def __init__(self, dataframe, processor, samples_per_stream=3, augment=False):
        self.processor = processor
        self.samples_per_stream = samples_per_stream
        self.augment = augment

        # Preload all audio into memory
        self.audio = []
        self.labels = []
        for _, row in dataframe.iterrows():
            wav, sr = torchaudio.load(row["abs_path"])
            if sr != TARGET_SR:
                wav = torchaudio.functional.resample(wav, sr, TARGET_SR)
            self.audio.append(wav.squeeze(0))
            self.labels.append(row["label_id"])
        print(f"  Loaded {len(self.audio)} streams, {len(self)} snippets")

    def __len__(self):
        return len(self.audio) * self.samples_per_stream

    def __getitem__(self, idx):
        stream_idx = idx // self.samples_per_stream
        wav = self.audio[stream_idx]

        # Random crop to WINDOW_SAMPLES
        if len(wav) <= WINDOW_SAMPLES:
            chunk = F.pad(wav, (0, WINDOW_SAMPLES - len(wav)))
        else:
            start = random.randint(0, len(wav) - WINDOW_SAMPLES)
            chunk = wav[start:start + WINDOW_SAMPLES]

        if self.augment:
            chunk = augment_waveform(chunk)

        # Force exact length (speed perturbation changes length)
        if len(chunk) > WINDOW_SAMPLES:
            chunk = chunk[:WINDOW_SAMPLES]
        elif len(chunk) < WINDOW_SAMPLES:
            chunk = F.pad(chunk, (0, WINDOW_SAMPLES - len(chunk)))

        inputs = self.processor(chunk, sampling_rate=TARGET_SR, return_tensors="pt", padding=False)
        return inputs.input_values.squeeze(0), self.labels[stream_idx]

def collate_fn(batch):
    wavs, labels = zip(*batch)
    wavs = torch.stack(wavs)  # All same length — no padding needed
    mask = torch.ones(wavs.shape, dtype=torch.bool)
    return wavs, mask, torch.tensor(labels, dtype=torch.long)

In [5]:
# wav2vec2-base-960h + freeze CNN + LoRA on attention
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
w2v = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h")

# Disable wav2vec2's built-in spec masking — the masked_spec_embed param is
# uninitialized (missing from 960h checkpoint) and filled with NaN/garbage.
# We apply our own SpecAugment on hidden states instead.
w2v.config.mask_time_prob = 0.0
w2v.config.mask_feature_prob = 0.0

# Freeze CNN feature extractor and feature projection
for param in w2v.feature_extractor.parameters():
    param.requires_grad = False
for param in w2v.feature_projection.parameters():
    param.requires_grad = False

# LoRA on transformer q_proj and v_proj
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)
w2v = get_peft_model(w2v, lora_config)
w2v.print_trainable_parameters()

# Classification head: dropout -> linear
class ClassificationHead(nn.Module):
    def __init__(self, hidden_size=768, num_labels=2, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.linear = nn.Linear(hidden_size, num_labels)

    def forward(self, x):
        return self.linear(self.dropout(x))

head = ClassificationHead()
w2v = w2v.to(DEVICE)
head = head.to(DEVICE)

lora_params = sum(p.numel() for p in w2v.parameters() if p.requires_grad)
head_params = sum(p.numel() for p in head.parameters())
print(f"LoRA params: {lora_params:,}  Head params: {head_params:,}  Total: {lora_params + head_params:,}")

Loading weights: 100%|██████████| 210/210 [00:00<00:00, 790.31it/s, Materializing param=feature_projection.projection.weight]                         
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.bias      | UNEXPECTED | 
lm_head.weight    | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 294,912 || all params: 94,666,624 || trainable%: 0.3115
LoRA params: 294,912  Head params: 1,538  Total: 296,450


In [6]:
BATCH_SIZE = 8
GRAD_ACCUM = 2  # effective batch = 16

print("Loading train set...")
train_ds = ConcatSpeechDataset(train_df, processor, samples_per_stream=SAMPLES_PER_STREAM, augment=True)
print("Loading val set...")
val_ds = ConcatSpeechDataset(val_df, processor, samples_per_stream=SAMPLES_PER_STREAM, augment=False)
print("Loading test set...")
test_ds = ConcatSpeechDataset(test_df, processor, samples_per_stream=SAMPLES_PER_STREAM, augment=False)

# WeightedRandomSampler for class balance (weights per snippet)
train_snippet_labels = np.array([train_ds.labels[i // SAMPLES_PER_STREAM] for i in range(len(train_ds))])
class_counts = np.bincount(train_snippet_labels)
sample_weights = (1.0 / class_counts)[train_snippet_labels]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(train_snippet_labels), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"\nTrain: {len(train_loader)} batches, Val: {len(val_loader)} batches, Test: {len(test_loader)} batches")

Loading train set...
  Loaded 299 streams, 897 snippets
Loading val set...
  Loaded 53 streams, 159 snippets
Loading test set...
  Loaded 88 streams, 264 snippets

Train: 113 batches, Val: 20 batches, Test: 33 batches


In [7]:
EPOCHS = 20

# Two param groups: LoRA lr=5e-5, head lr=1e-3
optimizer = torch.optim.AdamW([
    {"params": [p for p in w2v.parameters() if p.requires_grad], "lr": 5e-5},
    {"params": head.parameters(), "lr": 1e-3},
], weight_decay=0.01)

# Linear warmup 10% then linear decay
num_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS
num_warmup = int(0.1 * num_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup, num_steps)

criterion = nn.CrossEntropyLoss()  # unweighted; sampler handles balance

# SpecAugment on hidden states
freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=10)
time_mask = torchaudio.transforms.TimeMasking(time_mask_param=15)

print(f"Steps: {num_steps}, Warmup: {num_warmup}")

Steps: 1120, Warmup: 112


In [8]:
def mean_pool_with_mask(hidden, attn_mask, w2v_model):
    """Mean-pool hidden states using output-length mask (accounts for CNN downsampling)."""
    input_lengths = attn_mask.sum(dim=1)
    base = w2v_model.base_model.model if hasattr(w2v_model, "base_model") else w2v_model
    output_lengths = base._get_feat_extract_output_lengths(input_lengths).long()
    output_lengths = output_lengths.clamp(min=1)  # prevent division by zero
    out_mask = torch.arange(hidden.size(1), device=hidden.device).unsqueeze(0) < output_lengths.unsqueeze(1)
    pooled = (hidden * out_mask.unsqueeze(-1)).sum(1) / output_lengths.unsqueeze(1).float()
    return pooled

In [9]:
@torch.no_grad()
def evaluate(loader, w2v_model, head_model):
    """Evaluate and return metrics dict."""
    w2v_model.eval()
    head_model.eval()
    all_labels, all_preds, all_probs = [], [], []

    for wavs, mask, labels in loader:
        wavs, mask = wavs.to(DEVICE), mask.to(DEVICE)
        hidden = w2v_model(wavs, attention_mask=mask.float()).last_hidden_state
        pooled = mean_pool_with_mask(hidden, mask, w2v_model)
        logits = head_model(pooled)
        probs = torch.softmax(logits, dim=1)
        all_labels.extend(labels.tolist())
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_probs.extend(probs[:, 1].cpu().tolist())

    labels_arr = np.array(all_labels)
    preds_arr = np.array(all_preds)
    probs_arr = np.array(all_probs)
    f1_per = f1_score(labels_arr, preds_arr, average=None)

    return {
        "f1_macro": f1_score(labels_arr, preds_arr, average="macro"),
        "f1_ctrl": f1_per[0], "f1_dys": f1_per[1],
        "auc": roc_auc_score(labels_arr, probs_arr),
        "labels": labels_arr, "preds": preds_arr, "probs": probs_arr,
    }

In [10]:
best_f1 = 0.0
best_epoch = 0
best_w2v_state = None
best_head_state = None
history = []

for epoch in range(EPOCHS):
    w2v.train()
    head.train()
    optimizer.zero_grad()
    total_loss, correct, total = 0.0, 0, 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:2d}/{EPOCHS}", leave=False)
    for batch_idx, (wavs, mask, labels) in enumerate(pbar):
        wavs, mask, labels = wavs.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)

        hidden = w2v(wavs, attention_mask=mask.float()).last_hidden_state

        # SpecAugment on hidden states (treat as (B, 768, T) spectrogram)
        hidden_t = hidden.transpose(1, 2)
        hidden_t = freq_mask(hidden_t)
        hidden_t = time_mask(hidden_t)
        hidden = hidden_t.transpose(1, 2)

        pooled = mean_pool_with_mask(hidden, mask, w2v)
        logits = head(pooled)
        loss = criterion(logits, labels) / GRAD_ACCUM
        loss.backward()

        if (batch_idx + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(
                list(w2v.parameters()) + list(head.parameters()), max_norm=1.0
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * GRAD_ACCUM * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix(loss=f"{total_loss/total:.4f}", acc=f"{correct/total:.3f}")

    # Flush remaining gradients
    if (batch_idx + 1) % GRAD_ACCUM != 0:
        torch.nn.utils.clip_grad_norm_(
            list(w2v.parameters()) + list(head.parameters()), max_norm=1.0
        )
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    # Validate
    val = evaluate(val_loader, w2v, head)
    history.append({
        "epoch": epoch + 1, "train_loss": total_loss / total, "train_acc": correct / total,
        "val_f1": val["f1_macro"], "val_f1_ctrl": val["f1_ctrl"],
        "val_f1_dys": val["f1_dys"], "val_auc": val["auc"],
    })

    marker = ""
    if val["f1_macro"] > best_f1:
        best_f1 = val["f1_macro"]
        best_epoch = epoch + 1
        best_w2v_state = copy.deepcopy(w2v.state_dict())
        best_head_state = copy.deepcopy(head.state_dict())
        marker = "  ** best **"

    print(
        f"Epoch {epoch+1:2d}/{EPOCHS}  loss={total_loss/total:.4f}  acc={correct/total:.3f}  "
        f"val_F1={val['f1_macro']:.4f}  val_AUC={val['auc']:.4f}  "
        f"F1_ctrl={val['f1_ctrl']:.3f}  F1_dys={val['f1_dys']:.3f}{marker}"
    )

print(f"\nBest val F1 macro: {best_f1:.4f} at epoch {best_epoch}")

Epoch  1/20  loss=0.6929  acc=0.541  val_F1=0.5999  val_AUC=0.8710  F1_ctrl=0.738  F1_dys=0.462  ** best **


Epoch  2/20  loss=0.6845  acc=0.610  val_F1=0.3291  val_AUC=0.8558  F1_ctrl=0.000  F1_dys=0.658


Epoch  3/20  loss=0.6318  acc=0.677  val_F1=0.7149  val_AUC=0.9220  F1_ctrl=0.650  F1_dys=0.779  ** best **


Epoch  4/20  loss=0.5324  acc=0.763  val_F1=0.8930  val_AUC=0.9647  F1_ctrl=0.896  F1_dys=0.890  ** best **


Epoch  5/20  loss=0.4480  acc=0.800  val_F1=0.8397  val_AUC=0.9669  F1_ctrl=0.862  F1_dys=0.818


...

Epoch 16/20  loss=0.3014  acc=0.867  val_F1=0.9367  val_AUC=0.9989  F1_ctrl=0.942  F1_dys=0.932  ** best **


Epoch 17/20  loss=0.2799  acc=0.882  val_F1=0.9621  val_AUC=0.9995  F1_ctrl=0.964  F1_dys=0.960  ** best **


Epoch 18/20  loss=0.2671  acc=0.889  val_F1=0.9367  val_AUC=0.9992  F1_ctrl=0.942  F1_dys=0.932


Epoch 19/20  loss=0.2406  acc=0.893  val_F1=0.9303  val_AUC=0.9938  F1_ctrl=0.936  F1_dys=0.924


Epoch 20/20  loss=0.3149  acc=0.876  val_F1=0.9303  val_AUC=0.9934  F1_ctrl=0.936  F1_dys=0.924

Best val F1 macro: 0.9621 at epoch 17


In [11]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

MISC_DIR = os.path.join(BASE_DIR, "data", "misc")
os.makedirs(MISC_DIR, exist_ok=True)

h = pd.DataFrame(history)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(h["epoch"], h["train_loss"], "o-")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].set_title("Train Loss")

axes[1].plot(h["epoch"], h["val_f1"], "o-", label="F1 macro")
axes[1].plot(h["epoch"], h["val_f1_ctrl"], "s--", label="F1 ctrl")
axes[1].plot(h["epoch"], h["val_f1_dys"], "^--", label="F1 dys")
axes[1].axvline(best_epoch, color="red", ls=":", alpha=0.7, label=f"Best (ep {best_epoch})")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("F1"); axes[1].set_title("Val F1"); axes[1].legend()

axes[2].plot(h["epoch"], h["val_auc"], "o-", color="green")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("AUC-ROC"); axes[2].set_title("Val AUC-ROC")

fig.tight_layout()
fig.savefig(os.path.join(MISC_DIR, "lora_concat_training_curves.png"), dpi=150)
plt.close(fig)
print(f"Saved to {MISC_DIR}/lora_concat_training_curves.png")

Saved to /data/liharrison/lvsim/data/misc/lora_concat_training_curves.png


In [12]:
# Load best checkpoint and evaluate on test set
w2v.load_state_dict(best_w2v_state)
head.load_state_dict(best_head_state)
print(f"Loaded best checkpoint from epoch {best_epoch} (val F1={best_f1:.4f})")

test = evaluate(test_loader, w2v, head)
print(f"\n{'='*50}")
print(f"TEST SET  (epoch {best_epoch})")
print(f"{'='*50}")
print(f"F1 Macro:  {test['f1_macro']:.4f}")
print(f"AUC-ROC:   {test['auc']:.4f}")
print(f"F1 Ctrl:   {test['f1_ctrl']:.4f}")
print(f"F1 Dys:    {test['f1_dys']:.4f}")
print()
print(classification_report(test["labels"], test["preds"], target_names=["control", "dysfluent"]))

Loaded best checkpoint from epoch 17 (val F1=0.9621)

TEST SET  (epoch 17)
F1 Macro:  0.9773
AUC-ROC:   0.9982
F1 Ctrl:   0.9778
F1 Dys:    0.9767

              precision    recall  f1-score   support

     control       0.96      1.00      0.98       132
   dysfluent       1.00      0.95      0.98       132

    accuracy                           0.98       264
   macro avg       0.98      0.98      0.98       264
weighted avg       0.98      0.98      0.98       264



In [13]:
# Per-severity group accuracy on all streams
w2v.eval()
head.eval()
group_results = {s: {"correct": 0, "total": 0} for s in ["none", "moderate", "severe"]}

for severity, group_df in df.groupby("severity"):
    if severity not in group_results:
        continue
    ds = ConcatSpeechDataset(group_df, processor, samples_per_stream=SAMPLES_PER_STREAM, augment=False)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
    for wavs, mask, labels in loader:
        wavs, mask, labels = wavs.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)
        with torch.no_grad():
            hidden = w2v(wavs, attention_mask=mask.float()).last_hidden_state
            pooled = mean_pool_with_mask(hidden, mask, w2v)
            preds = head(pooled).argmax(1)
        group_results[severity]["correct"] += (preds == labels).sum().item()
        group_results[severity]["total"] += labels.size(0)

for sev, name in [("none", "control"), ("moderate", "moderate"), ("severe", "severe")]:
    r = group_results[sev]
    acc = r["correct"] / r["total"] if r["total"] > 0 else 0
    print(f"{name:>10s}  ({r['total']:3d} snippets)  acc={acc:.3f}")

  Loaded 110 streams, 330 snippets
  Loaded 220 streams, 660 snippets
  Loaded 110 streams, 330 snippets
   control  (660 snippets)  acc=0.998
  moderate  (330 snippets)  acc=0.852
    severe  (330 snippets)  acc=1.000


In [14]:
# Save LoRA adapter weights + classification head
save_dir = os.path.join(BASE_DIR, "data", "models")
os.makedirs(save_dir, exist_ok=True)

lora_path = os.path.join(save_dir, "lora_concat_w2v2_960h")
w2v.save_pretrained(lora_path)
print(f"LoRA adapters saved to: {lora_path}")

head_path = os.path.join(save_dir, "lora_concat_head.pt")
torch.save(head.state_dict(), head_path)
print(f"Head saved to: {head_path}")

LoRA adapters saved to: /data/liharrison/lvsim/data/models/lora_concat_w2v2_960h
Head saved to: /data/liharrison/lvsim/data/models/lora_concat_head.pt


In [15]:
# Test on real clinical data
REAL_DIR = os.path.join(BASE_DIR, "data", "real")
groups = {
    "lvPPA":      glob.glob(os.path.join(REAL_DIR, "*lvPPA*", "**", "*.wav"), recursive=True),
    "JHU":        glob.glob(os.path.join(REAL_DIR, "*jhu*", "**", "*.wav"), recursive=True),
    "Control":    glob.glob(os.path.join(REAL_DIR, "*segmentedcc*", "**", "*.wav"), recursive=True),
    "Capilouto":  glob.glob(os.path.join(REAL_DIR, "*Capilouto*", "**", "*.wav"), recursive=True),
}

w2v.eval()
head.eval()

for name, files in groups.items():
    probs = []
    for f in files:
        wav, sr = torchaudio.load(f)
        wav = torchaudio.functional.resample(wav, sr, 16000).mean(0)
        inputs = processor(wav, sampling_rate=16000, return_tensors="pt")
        input_values = inputs.input_values.to(DEVICE)
        attn_mask = torch.ones_like(input_values)
        with torch.no_grad():
            hidden = w2v(input_values, attention_mask=attn_mask).last_hidden_state
            pooled = mean_pool_with_mask(hidden, attn_mask, w2v)
            prob = torch.softmax(head(pooled), dim=1)[0, 1].item()
        probs.append(prob)
    n = len(probs)
    dys_50 = sum(p >= 0.5 for p in probs)
    dys_80 = sum(p >= 0.8 for p in probs)
    print(
        f"{name:>10s}  ({n:3d} clips)  avg_P={np.mean(probs):.3f}  "
        f"| t=0.5: ctrl={n-dys_50} dys={dys_50}  "
        f"| t=0.8: ctrl={n-dys_80} dys={dys_80}"
    )

     lvPPA  ( 89 clips)  avg_P=0.820  | t=0.5: ctrl=14 dys=75  | t=0.8: ctrl=22 dys=67
       JHU  ( 74 clips)  avg_P=0.924  | t=0.5: ctrl=4 dys=70  | t=0.8: ctrl=7 dys=67
   Control  (235 clips)  avg_P=0.257  | t=0.5: ctrl=176 dys=59  | t=0.8: ctrl=201 dys=34
 Capilouto  (311 clips)  avg_P=0.235  | t=0.5: ctrl=239 dys=72  | t=0.8: ctrl=261 dys=50


In [ ]:
# ── Reload saved model from scratch ──
from peft import PeftModel

processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
base_model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h")
model = PeftModel.from_pretrained(base_model, os.path.join(BASE_DIR, "data", "models", "lora_concat_w2v2_960h"))
model.eval().to(DEVICE)

head = ClassificationHead()
head.load_state_dict(torch.load(os.path.join(BASE_DIR, "data", "models", "lora_concat_head.pt"), weights_only=True))
head.eval().to(DEVICE)

print("Model reloaded successfully.")

In [16]:
# Distribution of P(dysfluent) on real data
REAL_DIR = os.path.join(BASE_DIR, "data", "real")
real_groups = {
    "lvPPA":     glob.glob(os.path.join(REAL_DIR, "*lvPPA*", "**", "*.wav"), recursive=True),
    "JHU":       glob.glob(os.path.join(REAL_DIR, "*jhu*", "**", "*.wav"), recursive=True),
    "Control":   glob.glob(os.path.join(REAL_DIR, "*segmentedcc*", "**", "*.wav"), recursive=True),
    "Capilouto": glob.glob(os.path.join(REAL_DIR, "*Capilouto*", "**", "*.wav"), recursive=True),
}

w2v.eval()
head.eval()
group_probs = {}

for name, files in real_groups.items():
    probs = []
    for f in files:
        wav, sr = torchaudio.load(f)
        wav = torchaudio.functional.resample(wav, sr, 16000).mean(0)
        inputs = processor(wav, sampling_rate=16000, return_tensors="pt")
        input_values = inputs.input_values.to(DEVICE)
        attn_mask = torch.ones_like(input_values)
        with torch.no_grad():
            hidden = w2v(input_values, attention_mask=attn_mask).last_hidden_state
            pooled = mean_pool_with_mask(hidden, attn_mask, w2v)
            prob = torch.softmax(head(pooled), dim=1)[0, 1].item()
        probs.append(prob)
    group_probs[name] = probs

fig, ax = plt.subplots(figsize=(8, 4))
bins = np.linspace(0, 1, 30)
for name, color in [("Control", "tab:blue"), ("Capilouto", "tab:cyan"), ("lvPPA", "tab:red"), ("JHU", "tab:orange")]:
    ax.hist(group_probs[name], bins=bins, alpha=0.5, label=f"{name} (n={len(group_probs[name])})", color=color, edgecolor="white")
ax.axvline(0.5, color="black", ls="--", alpha=0.5, label="Threshold 0.5")
ax.axvline(0.8, color="gray", ls=":", alpha=0.7, label="Threshold 0.8")
ax.set_xlabel("P(dysfluent)")
ax.set_ylabel("Count")
ax.set_title("Real Data: Distribution of P(dysfluent) — Concat Model")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(MISC_DIR, "real_data_prob_dist_concat.png"), dpi=150)
plt.close(fig)
print(f"Saved to {MISC_DIR}/real_data_prob_dist_concat.png")

Saved to /data/liharrison/lvsim/data/misc/real_data_prob_dist_concat.png
